# **Introduction:**
Our project aims to generate music from user prompts. We employed a two-step process to achieve this:

**Scene to Music Caption Generation:** We utilized the T5-large model to generate musical captions based on user-provided scenes. To train this model, we used a manually curated dataset consisting of scenes and corresponding musical captions.

**Music Generation from Captions:** Once we obtained the musical captions, we fed them into the Text2Midi model (which we cloned from GitHub). This model uses the FLAN-T5 encoder and a Transformer decoder to convert the text into music. Finally, FluidSynth was used to convert the generated MIDI to MP3 format, allowing us to play and download the final music.



# **Mount Google Drive**
In this step, we mount Google Drive to access our datasets and model files stored on it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# **Extract Model Files from Zip Archive**
In this step, we extract the T5-large model files from the ZIP archive stored on Google Drive. These files are required for loading the model to generate captions from the scene descriptions. The zip archive contains the necessary configuration, weights, and other model files, which are extracted to a specific directory for further use.


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/t5_large_scene_to_caption-20250414T082503Z-002.zip'
extract_path = '/content/t5_large_scene_to_caption'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


# **Install Required Libraries**
In this step, we install the essential libraries:

transformers: For working with pre-trained models like T5 and Flan-T5.

accelerate: For optimized training and inference.

safetensors: For efficiently loading model weights.

torch and torchvision: For deep learning functionality.

huggingface_hub: To access models from Hugging Face.

pretty_midi: To generate and manipulate MIDI files.

These libraries allow the system to effectively work with both the text-to-caption and text-to-MIDI models.

In [ ]:
!pip install transformers accelerate safetensors
!pip install torch torchvision huggingface_hub pretty_midi


# **Install MIDI Conversion Tools**
This cell installs the necessary tools for converting the generated MIDI to an audio format (MP3) that can be played and downloaded:

fluidsynth: A software synthesizer for converting MIDI to audio.

pydub: A Python library for audio processing, which will help convert the audio from WAV to MP3.

In [ ]:
# Install required tools
!apt-get install -y fluidsynth
!pip install pydub


# **Clone the Text2MIDI Repository and Install Dependencies**
In this step, we clone the Text2MIDI repository from GitHub. This repository contains the code for generating MIDI files from text captions, which we use to create music from scene descriptions. After cloning, we upgrade pip to the latest version and install the required dependencies listed in the requirements.txt file. These dependencies include the necessary libraries to run the Text2MIDI model.

In [ ]:
!git clone https://github.com/AMAAI-Lab/Text2midi.git
%cd Text2midi

!pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
!pip install --upgrade torchvision

In [ ]:
!pip install --upgrade torch

# **Import Libraries and Download SoundFont**
Here, we import the necessary libraries: pydub for audio manipulation, IPython.display for playing audio in Colab, and google.colab.files to facilitate file downloads. We also download the FluidR3 GM SoundFont if it is not already available. This SoundFont is used by FluidSynth to synthesize MIDI into audio.

In [ ]:
from pydub import AudioSegment
from IPython.display import Audio
from google.colab import files
import os
# Download a General MIDI SoundFont (if not already present)
soundfont_path = "FluidR3_GM.sf2"
if not os.path.exists(soundfont_path):
    !wget https://github.com/urish/cinto/raw/main/soundfonts/FluidR3_GM.sf2 -O FluidR3_GM.sf2

# **Load Scene-to-Caption Model**
In this step, we load the Scene-to-Caption model, which is a T5 large model fine-tuned to generate music captions based on scene descriptions. We also load the corresponding tokenizer and state dictionary. This model uses the T5 large configuration for text generation and is now ready for generating captions based on user input.

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config
from safetensors.torch import load_file
import torch

# Load config and model
config = T5Config.from_pretrained("t5-large")
model = T5ForConditionalGeneration(config)
state_dict = load_file("/content/drive/MyDrive/model.safetensors")
model.load_state_dict(state_dict, strict=False)
model.eval()

# For caption generation
tokenizer_caption = T5Tokenizer.from_pretrained("t5-large")

print("✅ Scene-to-Caption model loaded.")


In [ ]:
!mkdir model

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config

# **Load Text2MIDI Model**
This step loads the Text2MIDI model, which is designed to generate MIDI music from music captions. The model is based on a Transformer architecture and uses a REMI tokenizer for encoding the input text. We also check for the availability of CUDA for GPU acceleration. Additionally, we load the Flan-T5 tokenizer for tokenizing the generated captions into appropriate MIDI tokens. The model is now ready for generating MIDI music based on captions.

In [ ]:
import pickle
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from model.transformer_model import Transformer

repo_id = "amaai-lab/text2midi"
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load REMI tokenizer
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

# Load Text2MIDI model
vocab_size = len(r_tokenizer)
text2midi_model = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
text2midi_model.load_state_dict(torch.load(model_path, map_location=device))
text2midi_model.eval()
# For MIDI generation
tokenizer_midi = T5Tokenizer.from_pretrained("google/flan-t5-base")

print("✅ Text2MIDI model loaded.")


# **Generate Music from Scene Prompt**
This is the core execution cell of our project.

First, we take a scene description from the user.

Then, we use the T5-large caption model to generate a corresponding musical caption that reflects the scene's mood and energy.

This caption is passed into the Text2MIDI model, which translates it into MIDI token sequences.

Finally, the MIDI is decoded and saved as output.mid, ready to be played or converted into audio.

In [ ]:
# 🎬 Step 1: Get user input
scene_input = input("🎥 Enter a scene description: ")
tokenizer_caption = T5Tokenizer.from_pretrained("t5-large")
# 🎶 Step 2: Generate caption
inputs = tokenizer_caption(scene_input, return_tensors="pt")


with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_length=256,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

generated_caption = tokenizer_caption.decode(output_ids[0], skip_special_tokens=True)
print("\n🎵 Generated Music Caption:")
print(generated_caption)

# 🎼 Step 3: Generate MIDI
inputs = tokenizer_midi(generated_caption, return_tensors='pt', padding=True, truncation=True)
input_ids = nn.utils.rnn.pad_sequence(inputs.input_ids, batch_first=True, padding_value=0).to(device)
attention_mask = nn.utils.rnn.pad_sequence(inputs.attention_mask, batch_first=True, padding_value=0).to(device)

output = text2midi_model.generate(input_ids, attention_mask, max_len=2000, temperature=1.0)

# Decode and save
output_list = output[0].tolist()
generated_midi = r_tokenizer.decode(output_list)
generated_midi.dump_midi("output.mid")

print("\n✅ MIDI generated and saved as output.mid")


# **Convert and Play Music Output**
After generating the MIDI file, this step converts it into a high-quality MP3 audio file so that users can easily listen to and download the music.

We use Fluidsynth with a standard SoundFont to convert the MIDI to WAV.

Then, we convert WAV to MP3 using pydub.

Finally, we play the audio inline in Colab and offer a direct download link for the MP3 file.
This ensures the generated music is easily accessible and shareable.

In [ ]:
# Convert MIDI to WAV using fluidsynth
!fluidsynth -ni FluidR3_GM.sf2 output.mid -F output.wav -r 44100

# Convert WAV to MP3 using pydub
sound = AudioSegment.from_wav("output.wav")
sound.export("output.mp3", format="mp3")

# ✅ Play the MP3 in Colab
print("🎧 Here’s your generated music:")
audio = Audio("output.mp3")
display(audio)

# 📥 Provide a link for downloading the MP3 file
files.download("output.mp3")


# **🎯 Conclusion**
In this project, we built an end-to-end AI music generation system that transforms user-provided scene descriptions into expressive musical compositions.

We trained a T5-large model on a custom dataset of scene-to-music captions to understand and generate appropriate musical themes.

These captions were then passed to a Text2MIDI model, which used a Flan-T5 encoder and Transformer-based decoder to generate MIDI sequences.

Finally, we converted the generated MIDI to an MP3 format using Fluidsynth and Pydub, allowing users to play and download the audio directly from the notebook.

This pipeline showcases the power of combining NLP and music generation models to create emotionally relevant and context-aware music, opening up possibilities for applications in film scoring, game soundtracks, and creative media.

🔊✨ Thank you for exploring our project!